# FinanceBench Dataset Evaluation

In [7]:
%pip install -U transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
 

In [1]:
!pip install flash-attn --no-build-isolation # for A100 when we use flash-attn

  Using cached flash_attn-2.8.3.tar.gz (8.4 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [1]:
import torch
import json
import time
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import ast
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from difflib import SequenceMatcher
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

# %cd /kaggle/working/Active-Reading--Pattern-Recognition # # if in Kaggle
%cd /content/Active-Reading--Pattern-Recognition # if in Colab

os.getcwd()

Cloning into 'Active-Reading--Pattern-Recognition'...
remote: Enumerating objects: 3409, done.
remote: Counting objects: 100% (800/800), done.
remote: Compressing objects: 100% (727/727), done.
remote: Total 3409 (delta 132), reused 579 (delta 71), pack-reused 2609 (from 1)
Receiving objects: 100% (3409/3409), 102.03 MiB | 31.00 MiB/s, done.
Resolving deltas: 100% (797/797), done.
Filtering content: 100% (3/3), 315.19 MiB | 29.56 MiB/s, done.
[Errno 2] No such file or directory: '/content/Active-Reading--Pattern-Recognition # if in Colab'
/content


'/content'

In [2]:
%cd /content/Active-Reading--Pattern-Recognition

/content/Active-Reading--Pattern-Recognition


## Load the FinanceBench Dataset

In [3]:
print("Loading FinanceBench dataset...")
ds = load_dataset("PatronusAI/financebench")
print(f"Dataset loaded: {ds}")
print(f"\nDataset features: {ds['train'].features.keys()}")
print(f"\nTotal examples in train split: {len(ds['train'])}")

# Filter for Information Extraction entries only
print("\nFiltering for 'Information extraction' question_reasoning only...")
filtered_data = []
for example in ds['train']:
    if example.get('question_reasoning') == 'Information extraction':
        filtered_data.append(example)

print(f"Filtered examples: {len(filtered_data)} (from {len(ds['train'])} total)")

# Inspect a sample entry
if filtered_data:
    sample = filtered_data[0]
    print("\n--- Sample Entry ---")
    print(f"Company: {sample['company']}")
    print(f"Question: {sample['question']}")
    print(f"Answer: {sample['answer']}")
    print(f"Question Type: {sample['question_type']}")
    print(f"Question Reasoning: {sample['question_reasoning']}")

Loading FinanceBench dataset...
Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})

Dataset features: dict_keys(['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'])

Total examples in train split: 150

Filtering for 'Information extraction' question_reasoning only...
Filtered examples: 31 (from 150 total)

--- Sample Entry ---
Company: 3M
Question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Answer: $1577.00
Que

## Loading the model

In [4]:
# Model configuration
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# --- Check path to adapters ---
# ADAPTER_PATH = "/kaggle/working/Active-Reading--Pattern-Recognition/Fine-tuning-configurations/final_qlora_adapter_repet_financeb" # if in Kaggle
ADAPTER_PATH = "Fine-tuning-configurations/final_qlora_adapter_repet_financeb_2epochs" # if in Colab

print(f"Base model: {BASE_MODEL}")
print(f"Adapter path: {ADAPTER_PATH}")

Base model: Qwen/Qwen3-4B-Instruct-2507
Adapter path: Fine-tuning-configurations/final_qlora_adapter_repet_financeb_2epochs


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.float16, # for T4-GPU
    bnb_4bit_compute_dtype=torch.bfloat16, # for A100
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2", # for A100
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Model loaded successfully with adapter!")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded successfully with adapter!


## Evaluation Metrics

In [6]:
def validate_answer(model, tokenizer, question: str, expected_answer: str, predicted_answer: str) -> bool:
    """Use the model to validate if the predicted answer is correct."""
    validation_prompt = f"""You are an expert answer validator. Given a question, an expected answer, and a predicted answer, determine if the predicted answer is correct.

Question: {question}
Expected Answer: {expected_answer}
Predicted Answer: {predicted_answer}

Is the predicted answer correct? Answer with only YES or NO."""

    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": validation_prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )
    else:
        formatted_prompt = validation_prompt

    response = generate_text(
        model,
        tokenizer,
        prompt=formatted_prompt,
        max_tokens=16,
    )

    response = response.strip().upper()
    return "YES" in response

print("Model-based validator defined")

Model-based validator defined


## Generate Model Predictions

In [7]:
def generate_text(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    """
    Generate text using transformers model.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

def transformers_generate_compat(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    """
    Generate text using transformers model with chat template support.
    """
    if tokenizer.chat_template is not None:
        messages = [{"role": "user", "content": prompt}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )

    return generate_text(model, tokenizer, prompt, max_tokens)

# Configuration for evaluation
NUM_SAMPLES = 100  # Evaluate on first 100 samples; set to -1 for all
MAX_TOKENS = 256
PROMPT_TEMPLATE = """Answer the following question in great detail.

Question: {question}

Answer:"""

print(f"Evaluation configuration:")
print(f"- Number of samples: {NUM_SAMPLES if NUM_SAMPLES > 0 else len(filtered_data)}")
print(f"- Max tokens per generation: {MAX_TOKENS}")

Evaluation configuration:
- Number of samples: 100
- Max tokens per generation: 256


In [8]:
# Run evaluation
results = []
# num_eval_samples = NUM_SAMPLES if NUM_SAMPLES > 0 else len(filtered_data)
num_eval_samples = min(NUM_SAMPLES, len(filtered_data)) if NUM_SAMPLES > 0 else len(filtered_data)

print(f"\nGenerating predictions on {num_eval_samples} samples...")
print("=" * 80)

for idx in tqdm(range(num_eval_samples), desc="Evaluating"):
    example = filtered_data[idx]
    question = example['question']
    ground_truth = example['answer']
    # metadata = ast.literal_eval(example['metadata']) # finance bench doesn't support that
    # insted ... the following as metadata
    company = example.get('company', 'Unknown')
    doc_type = example.get('doc_type', 'Unknown')
    q_type = example.get('question_type', 'Unknown')

    # Generate prompt
    prompt = PROMPT_TEMPLATE.format(question=question)

    # Generate prediction
    start_time = time.time()
    prediction = transformers_generate_compat(model, tokenizer, prompt, MAX_TOKENS)
    generation_time = time.time() - start_time

    # Store results
    result = {
        'index': idx,
        'question': question,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'company': company,
        'doc_type': doc_type,
        'question_type': q_type,
        'generation_time': generation_time,
    }
    results.append(result)

print("=" * 80)
print(f"Predictions generated for {len(results)} samples")
print(f"Average generation time: {np.mean([r['generation_time'] for r in results]):.3f}s")


Generating predictions on 31 samples...


Evaluating: 100%|██████████| 31/31 [22:06<00:00, 42.80s/it]

Predictions generated for 31 samples
Average generation time: 42.800s


## Results

In [9]:
# Compute evaluation metrics for each result
print("Validating predictions using model-based validator...")

for result in tqdm(results, desc="Validating"):
    question = result['question']
    ground_truth = result['ground_truth']
    prediction = result['prediction']

    is_correct = validate_answer(model, tokenizer, question, ground_truth, prediction)
    result['is_correct'] = int(is_correct)

# Convert to DataFrame for easier analysis
df_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("OVERALL EVALUATION RESULTS")
print("=" * 80)
print(f"\nTotal samples evaluated: {len(df_results)}")
print(f"\nMetrics Summary:")
print(f"  Accuracy (Model-Validated): {df_results['is_correct'].mean():.4f} ({df_results['is_correct'].sum()}/{len(df_results)})")
print(f"  Avg Generation Time: {df_results['generation_time'].mean():.3f}s")

Validating predictions using model-based validator...


Validating: 100%|██████████| 31/31 [00:11<00:00,  2.68it/s]


OVERALL EVALUATION RESULTS

Total samples evaluated: 31

Metrics Summary:
  Accuracy (Model-Validated): 0.6774 (21/31)
  Avg Generation Time: 42.800s


In [10]:
# Save results to file
output_file = Path("financebench_evaluation_results.jsonl")
print(f"\nSaving results to {output_file}...")

with open(output_file, 'w') as f:
    for result in results:
        f.write(json.dumps(result) + '\n')

print(f"Results saved to {output_file}")


Saving results to financebench_evaluation_results.jsonl...
Results saved to financebench_evaluation_results.jsonl
